## Build Local LLM APPs with Ollama, DeepSeek-R1 and FAISS

In [1]:
!pip install purge --quiet

In [6]:
!pip install langchain langchain-community langchain-ollama --quiet

In [8]:
import re
import ollama
from langchain.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings
from langchain.vectorstores.utils import DistanceStrategy

In [9]:
embedding_model = "all-minilm"
ollama.pull(embedding_model)

ProgressResponse(status='success', completed=None, total=None, digest=None)

In [10]:
llm = "deepseek-r1:1.5b"
ollama.pull(llm)

ProgressResponse(status='success', completed=None, total=None, digest=None)

In [13]:
documents = [
    "Llamas are members of the camelid family meaning they're pretty closely related to vicuñas and camels",
    "Llamas were first domesticated and used as pack animals 4,000 to 5,000 years ago in the Peruvian highlands",
    "Llamas can grow as much as 6 feet tall though the average llama between 5 feet 6 inches and 5 feet 9 inches tall",
    "Llamas weigh between 280 and 450 pounds and can carry 25 to 30 percent of their body weight",
    "Llamas are vegetarians and have very efficient digestive systems",
    "Llamas live to be about 20 years old, though some only live for 15 years and others live to be 30 years old"
]

embeddings = OllamaEmbeddings(
    model = embedding_model,
)
print(len(documents[0]))

dimensions = len(embeddings.embed_query(documents[0]))
docs = [Document(text) for text in documents]

print("Dimensions: ", dimensions)
print("Docs are :\n",docs)

101
Dimensions:  384
Docs are :
 [Document(page_content="Llamas are members of the camelid family meaning they're pretty closely related to vicuñas and camels"), Document(page_content='Llamas were first domesticated and used as pack animals 4,000 to 5,000 years ago in the Peruvian highlands'), Document(page_content='Llamas can grow as much as 6 feet tall though the average llama between 5 feet 6 inches and 5 feet 9 inches tall'), Document(page_content='Llamas weigh between 280 and 450 pounds and can carry 25 to 30 percent of their body weight'), Document(page_content='Llamas are vegetarians and have very efficient digestive systems'), Document(page_content='Llamas live to be about 20 years old, though some only live for 15 years and others live to be 30 years old')]


FAISS

In [16]:
docsearch = FAISS.from_documents(docs,embeddings)

In [20]:
prompt = "What animals are llamas related to?"
docs = docsearch.similarity_search(prompt)
data = docs[0].page_content
print(f"For the Propmt {prompt} \nThe data found is \n{data}")


For the Propmt What animals are llamas related to? 
The data found is 
Llamas are members of the camelid family meaning they're pretty closely related to vicuñas and camels


In [21]:
prompt = "What is the life span of llamas?"
docs = docsearch.similarity_search(prompt)
data = docs[0].page_content
print(f"For the Propmt {prompt} \nThe data found is \n{data}")

For the Propmt What is the life span of llamas? 
The data found is 
Llamas live to be about 20 years old, though some only live for 15 years and others live to be 30 years old


In [23]:
output = ollama.generate(
    model = llm,
    prompt = f"Using this data {data}. Respond to this Prompt: \n{prompt}"
)
content = output["response"]
remove_think_tags = True

if remove_think_tags:
    content = re.sub("<think>.*?</think>", "", content, flags = re.DOTALL)

print(content)



The life span of llamas varies among individuals. On average, they live around 20 years when considering their typical lifespan. However, individual llamas can reach as short as 15 years or as long as up to 30 years before dying. This variation is due to differences in diet, disease, and environmental conditions affecting different llama strains.


In [ ]:
Llamas are tailless tail Animals belonging to the Largilphas family. 
They share a lineage with vicuñas, which are also part of the camelid family, 
indicating close relatedness. However, unlike vicuñas, 
llamas do not possess tails, distinguishing them from other tailless tail Animals like the wild boars.